In [1]:
!nvidia-smi

Tue Sep  1 15:59:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q -U datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.2 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import load_dataset
from trl import SFTTrainer
import transformers

# We will use a smaller instruction model for demonstration
model_id = "NousResearch/Llama-2-7b-chat-hf"

# Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load Model and Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"":0}
)

# Prepare model for PEFT training (enables gradient checkpointing)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [4]:
config = LoraConfig(
    r=8, # Rank
    lora_alpha=32, # Scaling factor
    target_modules=["q_proj", "v_proj"], # Apply LoRA to attention mechanisms
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap the base model with PEFT
model = get_peft_model(model, config)
model.print_trainable_parameters()
# Output will show ~0.1% of total parameters are trainable!

trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import transformers

# 1. Dataset load karo
dataset = load_dataset("Abirate/english_quotes")

# 2. Dataset ne pehlathi j format kari lo (Error-free method)
def format_row(row):
    row["text"] = f"### Quote: {row['quote']}\n ### Author: {row['author']}"
    return row

# Train dataset par mapping apply karo
mapped_dataset = dataset["train"].map(format_row)



# 1. Training configuration
sft_config = SFTConfig(
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=2,
    max_steps=50,
    learning_rate=2e-4,
    fp16=False,      # T4 GPU mate fp16
    bf16=True,       # bf16 bandh karyu
    logging_steps=10,
    output_dir="outputs",
    optim="paged_adamw_8bit"
)

# 2. SFTTrainer initialize karo
trainer = SFTTrainer(
    model=model,
    train_dataset=mapped_dataset,
    args=sft_config,
    processing_class=tokenizer,
)

model.config.use_cache = False
trainer.train()

Step,Training Loss
10,2.058879
20,1.394589
30,1.413722
40,1.306247
50,1.033006


TrainOutput(global_step=50, training_loss=1.441288661956787, metrics={'train_runtime': 376.0789, 'train_samples_per_second': 2.127, 'train_steps_per_second': 0.133, 'total_flos': 3358079448023040.0, 'train_loss': 1.441288661956787, 'epoch': 0.3189792663476874})

In [14]:
model.eval()

prompt = "### Quote: Be yourself; everyone else is already taken.\n ### Author:"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate response
outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Quote: Be yourself; everyone else is already taken.
 ### Author: Oscar Wilde


In [16]:
model.save_pretrained("my-awesome-lora-adapter")
# Now your deliverable is complete!

In [17]:
# Tame je pan quote/prashna pucho te ahiya lakho:
my_input = "In the middle of difficulty lies opportunity."

# Model mate prompt format karo
prompt = f"### Quote: {my_input}\n ### Author:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Model javab generate karshe
outputs = model.generate(
    **inputs,
    max_new_tokens=30,
    temperature=0.7,
    repetition_penalty=1.1
)

# Javab print karo
full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(full_response)

### Quote: In the middle of difficulty lies opportunity.
 ### Author: Albert Einstein
